# Migration Validation Metrics (DPC1.0 → DPC2.0)

Collects metadata metrics for the Hive tables on both platforms and compares them, using the
same checks the NX1 MapR/HDFS → S3 migration DAG runs after each copy. It works no matter how the
data was copied. The notebook is **read-only** against the tables: it only runs `SHOW`,
`DESCRIBE`, `SELECT COUNT(*)` and storage listings. The only files it writes are the snapshot and
report files under `OUTPUT_DIR`.

### What is checked per table

| Metric | How it is collected | Mismatch → |
|---|---|---|
| Row count (total and per partition) | `COUNT(*)` grouped by the partition columns | **FAIL** |
| Schema: non-partition columns, name + type | `spark.table(...).schema` | **FAIL** |
| Partition columns, name + type | `DESCRIBE TABLE EXTENDED` + schema | **FAIL** |
| Registered partitions | `SHOW PARTITIONS` | WARN |
| Data files and bytes (total and per partition) | listing under the table `LOCATION`, skipping hidden files (`_SUCCESS`, `.crc`, `_temporary`, …) | WARN |
| Partition directories that hold data but are not registered | storage listing vs `SHOW PARTITIONS` | WARN (fix with `MSCK REPAIR TABLE`) |

Other statuses: `MISSING_ON_DEST` (table not found on DPC2.0), `EXTRA_ON_DEST` (table only on
DPC2.0), `ERROR` (metrics could not be collected), `SKIPPED_VIEW` (views hold no data).

### How to run

**If one Spark session can see both platforms** (for example, a second catalog points at the other
metastore): keep `SIDES_TO_COLLECT = ["source", "dest"]` and run all cells.

**If the platforms are separate**, `OUTPUT_DIR` must be readable from both:
1. On the DPC1.0 JupyterHub, set `SIDES_TO_COLLECT = ["source"]` and run all cells. This writes `source_metrics.json`.
2. On the DPC2.0 JupyterHub, set `SIDES_TO_COLLECT = ["dest"]` and run all cells. This writes `dest_metrics.json`, then compares it with the source snapshot.

Each collection saves its metrics to `source_metrics.json` / `dest_metrics.json` under
`OUTPUT_DIR/RUN_LABEL`, and the comparison only reads those files. To compare again without
re-collecting (e.g. to regenerate the report), set `SIDES_TO_COLLECT = []`.

### Outputs

```
OUTPUT_DIR/RUN_LABEL/
├── source_metrics.json             # raw metrics, one record per table
├── dest_metrics.json
└── report_<UTC timestamp>/
    ├── summary.csv                 # one row per table, with status and every metric
    ├── partition_diffs.csv         # every partition that differs
    └── report.html
```

### Limits

- Storage metrics only cover files under the table `LOCATION`. Row counts and `SHOW PARTITIONS` still
  include partitions that live elsewhere.
- On Iceberg tables, the storage listing also counts files left behind by expired snapshots that
  have not been cleaned up yet, and `SHOW PARTITIONS` is skipped.
- Metrics are a point-in-time snapshot. If tables are still being written to, collect both sides
  again close together.
- `COUNT(*)` on Parquet/ORC is answered from file footers. Text tables are scanned in full.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────────
import os

# Groups the snapshots and reports of one validation exercise under OUTPUT_DIR/RUN_LABEL.
RUN_LABEL = "dpc1-to-dpc2"

# An s3a:// path readable from both platforms, or a local directory. With a local directory on
# two separate JupyterHubs, copy source_metrics.json across before running the comparison.
OUTPUT_DIR = os.environ.get("MIGRATION_VALIDATION_DIR", "s3a://CHANGE_ME/migration-validation")

# One session that sees both platforms: keep both. Separate platforms: ["source"] on DPC1.0,
# then ["dest"] on DPC2.0. The comparison runs wherever both snapshots are readable.
SIDES_TO_COLLECT = ["source", "dest"]

# "catalog" is prefixed to every table name; leave it empty for the session's default catalog.
SOURCE = {"label": "DPC1.0", "catalog": ""}
DEST = {"label": "DPC2.0", "catalog": ""}

# tables / exclude: glob patterns matched against table names (case-insensitive).
# partition_filter: optional Spark SQL predicate on partition columns, applied to both sides.
# Use it when only part of a table was migrated, e.g. "dt >= '2024-01-01'".
DATABASE_PAIRS = [
    {"source_db": "CHANGE_ME", "dest_db": "CHANGE_ME", "tables": ["*"], "exclude": [], "partition_filter": ""},
]

# Lists every data file under each table location. If that is too slow for very large tables,
# turn it off: row counts, partitions and schema are still compared.
COLLECT_STORAGE_METRICS = True
PARALLELISM = 8  # tables collected concurrently
REPORT_MAX_PARTITION_ROWS = 500  # cap for the HTML report only; partition_diffs.csv has every row

assert "CHANGE_ME" not in OUTPUT_DIR, "Set OUTPUT_DIR (or the MIGRATION_VALIDATION_DIR env var)"
assert all("CHANGE_ME" not in p["source_db"] + p["dest_db"] for p in DATABASE_PAIRS), "Set DATABASE_PAIRS"
assert set(SIDES_TO_COLLECT) <= {"source", "dest"}, f"Unknown side in {SIDES_TO_COLLECT}"

---
## Helpers

Nothing in this section touches the tables. Run it, then move on.

In [ ]:
# ── Spark session, storage I/O, naming ──────────────────────────────────────────────
import fnmatch
import html
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from urllib.parse import unquote

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, StructField, StructType

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder.appName("migration-validation-metrics").enableHiveSupport().getOrCreate()
    print(f"Created Spark session (version {spark.version})")

# A partition_filter is pushed down to the Hive metastore, which rejects it on DATE partition
# keys; this makes Spark list the partitions and filter them itself instead.
spark.conf.set("spark.sql.hive.metastorePartitionPruningFallbackOnException", "true")

HIVE_NULL = "__HIVE_DEFAULT_PARTITION__"

OUTPUT_ROOT = re.sub(r"^s3n?://", "s3a://", OUTPUT_DIR.rstrip("/"))
if "://" not in OUTPUT_ROOT:
    OUTPUT_ROOT = "file://" + os.path.abspath(OUTPUT_ROOT)
RUN_DIR = f"{OUTPUT_ROOT}/{RUN_LABEL}"


def snapshot_path(side):
    return f"{RUN_DIR}/{side}_metrics.json"


def _hpath(path):
    return spark._jvm.org.apache.hadoop.fs.Path(path)


def _fs(hpath):
    return hpath.getFileSystem(spark._jsc.hadoopConfiguration())


# Snapshots and reports go through the Hadoop FileSystem on the driver, so s3a:// and local
# paths work the same way and no executor ever needs to see a local path.
def write_text(path, text):
    hp = _hpath(path)
    out = _fs(hp).create(hp, True)
    try:
        out.write(bytearray(text.encode("utf-8")))
    finally:
        out.close()


def read_text(path):
    hp = _hpath(path)
    fs = _fs(hp)
    if not fs.exists(hp):
        return None
    buf = spark._jvm.java.io.ByteArrayOutputStream()
    stream = fs.open(hp)
    try:
        spark._jvm.org.apache.hadoop.io.IOUtils.copyBytes(stream, buf, 1 << 16, False)
    finally:
        stream.close()
    return buf.toString("UTF-8")


def quote_ident(name):
    return "`" + name.replace("`", "``") + "`"


def table_ident(catalog, db, table=None):
    return ".".join(quote_ident(p) for p in (catalog, db, table) if p)


def pair_id(pair):
    pf = pair.get("partition_filter") or ""
    return f"{pair['source_db']} -> {pair['dest_db']}" + (f" [{pf}]" if pf else "")


def short_error(exc, limit=500):
    msg = str(exc).strip()
    return (msg.splitlines()[0] if msg else type(exc).__name__)[:limit]


def fmt_bytes(n):
    if n is None or pd.isna(n):
        return ""
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or unit == "TB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.2f} {unit}"
        n /= 1024


print(f"Snapshots and reports: {RUN_DIR}")

In [ ]:
# ── Partition keys ──────────────────────────────────────────────────────────────────
# SHOW PARTITIONS and partition directory names share Hive's escaping (e.g. "a%2Fb"), so keys
# from those two sources are compared escaped. Values are only unescaped for partition_filter.


def normalize_partition_spec(spec):
    segments = []
    for seg in spec.strip().strip("/").split("/"):
        key, sep, value = seg.partition("=")
        if sep:
            segments.append(f"{key.lower()}={value}")
    return "/".join(segments)


def filter_partition_keys(keys, part_cols, predicate):
    """Keep the partition keys that satisfy a Spark SQL predicate on the partition columns."""
    keys = list(keys)
    if not predicate or not keys:
        return set(keys)
    rows = []
    for key in keys:
        values = {}
        for seg in key.split("/"):
            col, _, value = seg.partition("=")
            value = unquote(value)
            values[col] = None if value == HIVE_NULL else value
        rows.append(tuple([key] + [values.get(c) for c in part_cols]))
    schema = StructType([StructField("__key", StringType())] + [StructField(c, StringType()) for c in part_cols])
    return {r["__key"] for r in spark.createDataFrame(rows, schema).where(predicate).select("__key").collect()}

---
## Metric collection

`collect_table` gathers the metrics for one table. For the reasoning behind each one, see the
table at the top of the notebook.

In [ ]:
# ── Per-table metrics ───────────────────────────────────────────────────────────────


def describe_table(name):
    info = {"partition_columns": [], "partition_transforms": []}
    section = "columns"
    for row in spark.sql(f"DESCRIBE TABLE EXTENDED {name}").collect():
        key = (row.col_name or "").strip()
        value = (row.data_type or "").strip()
        if key.startswith("#"):
            if key == "# Partition Information":
                section = "hive_partitions"
            elif key == "# Partitioning":
                section = "v2_partitions"
            elif key != "# col_name":
                section = "details"
            continue
        if not key:
            continue
        if section == "hive_partitions":
            info["partition_columns"].append(key.lower())
        elif section == "v2_partitions":
            # Iceberg lists identity partitions as a bare column, others as e.g. "days(ts)".
            if re.fullmatch(r"`?[\w.]+`?", value):
                info["partition_columns"].append(value.strip("`").lower())
            else:
                info["partition_transforms"].append(value)
        elif section == "details":
            info.setdefault(key.lower(), value)
    return info


def count_rows(name, part_cols, predicate):
    where = f" WHERE {predicate}" if predicate else ""
    if not part_cols:
        count = spark.sql(f"SELECT COUNT(*) AS c FROM {name}{where}").first()["c"]
        return {"row_count": count, "partition_rows": None}
    # The grouped count costs the same as a plain COUNT(*) and also gives rows per partition.
    exprs = [f"CAST({quote_ident(c)} AS STRING)" for c in part_cols]
    select = ", ".join(f"{e} AS p{i}" for i, e in enumerate(exprs))
    partition_rows = {}
    for r in spark.sql(f"SELECT {select}, COUNT(*) AS c FROM {name}{where} GROUP BY {', '.join(exprs)}").collect():
        key = "/".join(f"{c}={HIVE_NULL if r[f'p{i}'] is None else r[f'p{i}']}" for i, c in enumerate(part_cols))
        partition_rows[key] = r["c"]
    return {"row_count": sum(partition_rows.values()), "partition_rows": partition_rows}


def registered_partitions(name, part_cols, predicate):
    specs = {normalize_partition_spec(r[0]) for r in spark.sql(f"SHOW PARTITIONS {name}").collect()}
    return sorted(filter_partition_keys(specs, part_cols, predicate))


def storage_metrics(location, part_cols, provider, predicate):
    root = _hpath(location)
    fs = _fs(root)
    if not fs.exists(root):
        return {"location_exists": False, "data_file_count": 0, "data_bytes": 0, "storage_partitions": {}}
    root_path = fs.makeQualified(root).toUri().getPath().rstrip("/")
    depth = len(part_cols)
    per_partition = {}
    loose_files = loose_bytes = 0
    files = fs.listFiles(root, True)
    while files.hasNext():
        status = files.next()
        segs = status.getPath().toUri().getPath()[len(root_path):].strip("/").split("/")
        # Same rule Hive and Spark use when reading: anything under a _ or . name is not data.
        if any(s.startswith(("_", ".")) for s in segs):
            continue
        if provider == "iceberg":
            if segs[0] == "metadata":
                continue
            if segs[0] == "data":
                segs = segs[1:]
        dirs = segs[:-1]
        if depth and len(dirs) >= depth and all("=" in d for d in dirs[:depth]):
            stats = per_partition.setdefault(normalize_partition_spec("/".join(dirs[:depth])), [0, 0])
            stats[0] += 1
            stats[1] += status.getLen()
        else:
            loose_files += 1
            loose_bytes += status.getLen()
    if predicate:
        keep = filter_partition_keys(per_partition, part_cols, predicate)
        per_partition = {k: v for k, v in per_partition.items() if k in keep}
        loose_files = loose_bytes = 0
    return {
        "location_exists": True,
        "data_file_count": loose_files + sum(v[0] for v in per_partition.values()),
        "data_bytes": loose_bytes + sum(v[1] for v in per_partition.values()),
        "files_outside_partition_dirs": loose_files,
        "storage_partitions": per_partition,
    }


def collect_table(pid, catalog, db, table, partition_filter):
    name = table_ident(catalog, db, table)
    rec = {"pair": pid, "database": db, "table": table, "status": "OK", "error": None, "notes": []}
    started = time.time()
    try:
        info = describe_table(name)
        rec["table_type"] = (info.get("type") or "").upper()
        rec["provider"] = (info.get("provider") or "").lower()
        rec["location"] = info.get("location") or ""
        if "VIEW" in rec["table_type"]:
            rec["status"] = "VIEW"
            return rec

        part_cols = info["partition_columns"]
        transforms = info["partition_transforms"]
        if transforms:
            rec["notes"].append(
                f"non-identity partitioning {transforms}: partition-level metrics cover identity columns only"
            )
        fields = spark.table(name).schema.fields
        types = {f.name.lower(): f.dataType.simpleString() for f in fields}
        rec["partition_columns"] = part_cols
        rec["schema"] = [{"name": n, "type": t} for n, t in types.items() if n not in part_cols]
        rec["partition_schema"] = [{"name": c, "type": types.get(c, "string")} for c in part_cols]

        predicate = partition_filter if (part_cols or transforms) else ""
        if partition_filter and not predicate:
            rec["notes"].append("partition_filter ignored: table is not partitioned")
        rec["partition_filter"] = predicate

        rec.update(count_rows(name, part_cols, predicate))

        rec["registered_partitions"] = None
        if part_cols and rec["provider"] != "iceberg":
            try:
                rec["registered_partitions"] = registered_partitions(name, part_cols, predicate)
            except Exception as e:
                rec["notes"].append(f"SHOW PARTITIONS failed: {short_error(e, 200)}")

        if COLLECT_STORAGE_METRICS and rec["location"] and predicate and not part_cols:
            rec["notes"].append("storage metrics skipped: partition_filter cannot be mapped to directories")
        elif COLLECT_STORAGE_METRICS and rec["location"]:
            rec.update(storage_metrics(rec["location"], part_cols, rec["provider"], predicate))
            if not rec["location_exists"]:
                rec["notes"].append(f"table location does not exist: {rec['location']}")
            if rec["registered_partitions"] is not None:
                registered = set(rec["registered_partitions"])
                rec["unregistered_partitions"] = sorted(k for k in rec["storage_partitions"] if k not in registered)
    except Exception as e:
        rec["status"] = "ERROR"
        rec["error"] = short_error(e)
    finally:
        rec["collect_seconds"] = round(time.time() - started, 1)
    return rec


def describe_record(rec):
    label = f"{rec['database']}.{rec['table']}"
    if rec["status"] != "OK":
        return f"{label}: {rec['status']} {rec.get('error') or ''}".rstrip()
    parts = [f"rows={rec['row_count']:,}"]
    if rec.get("registered_partitions") is not None:
        parts.append(f"partitions={len(rec['registered_partitions']):,}")
    if "data_file_count" in rec:
        parts.append(f"files={rec['data_file_count']:,} ({fmt_bytes(rec['data_bytes'])})")
    return f"{label}: {' '.join(parts)} [{rec['collect_seconds']}s]"

In [ ]:
# ── Per-side collection ─────────────────────────────────────────────────────────────


def resolve_tables(catalog, db, includes, excludes):
    rows = spark.sql(f"SHOW TABLES IN {table_ident(catalog, db)}").collect()
    names = sorted({r["tableName"] for r in rows if not r.asDict().get("isTemporary")})

    def matches(name, patterns):
        return any(fnmatch.fnmatch(name.lower(), p.lower()) for p in patterns)

    return [n for n in names if matches(n, includes) and not matches(n, excludes)]


def collect_side(side):
    cfg = SOURCE if side == "source" else DEST
    db_key = "source_db" if side == "source" else "dest_db"
    records, jobs = [], []
    for pair in DATABASE_PAIRS:
        pid, db = pair_id(pair), pair[db_key]
        try:
            tables = resolve_tables(cfg["catalog"], db, pair.get("tables") or ["*"], pair.get("exclude") or [])
        except Exception as e:
            records.append({"pair": pid, "database": db, "table": None, "status": "ERROR",
                            "error": f"cannot list tables in {db}: {short_error(e)}"})
            print(f"[{side}] {db}: cannot list tables: {short_error(e)}")
            continue
        print(f"[{side}] {db}: {len(tables)} table(s)")
        jobs += [(pid, cfg["catalog"], db, t, pair.get("partition_filter") or "") for t in tables]

    with ThreadPoolExecutor(max_workers=PARALLELISM) as pool:
        futures = [pool.submit(collect_table, *job) for job in jobs]
        for i, fut in enumerate(as_completed(futures), 1):
            rec = fut.result()
            records.append(rec)
            print(f"  [{side} {i}/{len(jobs)}] {describe_record(rec)}")

    snapshot = {
        "side": side,
        "label": cfg["label"],
        "catalog": cfg["catalog"],
        "run_label": RUN_LABEL,
        "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "spark_version": spark.version,
        "database_pairs": DATABASE_PAIRS,
        "tables": records,
    }
    write_text(snapshot_path(side), json.dumps(snapshot, indent=1, default=str))
    errors = sum(r["status"] == "ERROR" for r in records)
    print(f"[{side}] {len(records)} record(s), {errors} error(s) -> {snapshot_path(side)}\n")
    return snapshot

---
## Step 1: Collect metrics

Collects the metrics for each side in `SIDES_TO_COLLECT` and writes them to `<side>_metrics.json`.
Running it again overwrites the previous snapshot for that side.

In [ ]:
for _side in SIDES_TO_COLLECT:
    collect_side(_side)

---
## Step 2: Compare

Reads both snapshots back from `OUTPUT_DIR` and compares them table by table. Tables are
matched by name within each database pair.

In [ ]:
# ── Comparison ──────────────────────────────────────────────────────────────────────


def compare_columns(src, dest, kind):
    s = {c["name"]: c["type"].lower() for c in src or []}
    d = {c["name"]: c["type"].lower() for c in dest or []}
    diffs = []
    for name, stype in s.items():
        if name not in d:
            diffs.append(f"Missing {kind} on dest: {name}")
        elif d[name] != stype:
            diffs.append(f"Type mismatch for {kind} {name}: {stype} (source) vs {d[name]} (dest)")
    diffs += [f"Extra {kind} on dest: {name}" for name in d if name not in s]
    return diffs


def partition_diffs(s, d):
    out = []
    src_rows, dest_rows = s.get("partition_rows"), d.get("partition_rows")
    if src_rows is not None and dest_rows is not None:
        for key in sorted(set(src_rows) | set(dest_rows)):
            sr, dr = src_rows.get(key), dest_rows.get(key)
            if sr != dr:
                issue = (
                    "ROWS_MISSING_ON_DEST" if dr is None
                    else "ROWS_EXTRA_ON_DEST" if sr is None
                    else "ROW_COUNT_MISMATCH"
                )
                out.append({"partition": key, "issue": issue, "source_rows": sr, "dest_rows": dr})

    src_files, dest_files = s.get("storage_partitions"), d.get("storage_partitions")
    if src_files is not None and dest_files is not None:
        for key in sorted(set(src_files) | set(dest_files)):
            sf, df = src_files.get(key), dest_files.get(key)
            if sf != df:
                issue = (
                    "FILES_MISSING_ON_DEST" if df is None
                    else "FILES_EXTRA_ON_DEST" if sf is None
                    else "FILE_STATS_MISMATCH"
                )
                out.append({
                    "partition": key, "issue": issue,
                    "source_files": sf[0] if sf else None, "dest_files": df[0] if df else None,
                    "source_bytes": sf[1] if sf else None, "dest_bytes": df[1] if df else None,
                })

    for side, rec in (("source", s), ("dest", d)):
        out += [
            {"partition": key, "issue": f"NOT_REGISTERED_ON_{side.upper()}"}
            for key in rec.get("unregistered_partitions") or []
        ]
    return out


def compare_table(s, d):
    """Returns (summary fields, partition diff rows) for one table present on at least one side."""
    if s and s["status"] == "VIEW":
        return {"status": "SKIPPED_VIEW", "issues": "source is a view"}, []
    if s is None:
        return {"status": "EXTRA_ON_DEST", "issues": "table exists only on dest"}, []
    if s["status"] == "ERROR":
        return {"status": "ERROR", "issues": f"source: {s['error']}"}, []
    if d is None:
        return {"status": "MISSING_ON_DEST", "issues": "table not found on dest"}, []
    if d["status"] == "ERROR":
        return {"status": "ERROR", "issues": f"dest: {d['error']}"}, []
    if d["status"] == "VIEW":
        return {"status": "FAIL", "issues": "source is a table but dest is a view"}, []

    fails, warns = [], []
    if s["row_count"] != d["row_count"]:
        fails.append(f"Row count mismatch: source={s['row_count']:,}, dest={d['row_count']:,}")
    schema_diffs = compare_columns(s.get("schema"), d.get("schema"), "column")
    part_schema_diffs = compare_columns(s.get("partition_schema"), d.get("partition_schema"), "partition column")
    if schema_diffs:
        fails.append("Schema: " + "; ".join(schema_diffs[:3]) + (" ..." if len(schema_diffs) > 3 else ""))
    if part_schema_diffs:
        fails.append("Partition schema: " + "; ".join(part_schema_diffs[:3]))

    sp, dp = s.get("registered_partitions"), d.get("registered_partitions")
    if sp is not None and dp is not None and len(sp) != len(dp):
        warns.append(f"Partition count mismatch: source={len(sp):,}, dest={len(dp):,}")
    if d.get("unregistered_partitions"):
        warns.append(
            f"{len(d['unregistered_partitions']):,} partition dir(s) with data on dest are not registered: "
            "run MSCK REPAIR TABLE"
        )
    if "data_file_count" in s and "data_file_count" in d:
        if s["data_file_count"] != d["data_file_count"]:
            warns.append(f"Data file count mismatch: source={s['data_file_count']:,}, dest={d['data_file_count']:,}")
        if s["data_bytes"] != d["data_bytes"]:
            warns.append(f"Data size mismatch: source={s['data_bytes']:,} B, dest={d['data_bytes']:,} B")

    summary = {
        "status": "FAIL" if fails else "WARN" if warns else "PASS",
        "issues": "; ".join(fails + warns),
        "schema_differences": "; ".join(schema_diffs),
        "partition_schema_differences": "; ".join(part_schema_diffs),
    }
    return summary, partition_diffs(s, d)


def compare_snapshots(src_snap, dest_snap):
    if src_snap["database_pairs"] != dest_snap["database_pairs"]:
        print("WARNING: DATABASE_PAIRS differed between the two collections; tables may show as missing/extra.")

    def index(snap):
        tables, db_errors = {}, {}
        for r in snap["tables"]:
            if r["table"] is None:
                db_errors[r["pair"]] = r["error"]
            else:
                tables[(r["pair"], r["table"].lower())] = r
        return tables, db_errors

    src, src_db_errors = index(src_snap)
    dest, dest_db_errors = index(dest_snap)

    summary_rows, diff_rows = [], []
    for pid, err in src_db_errors.items():
        summary_rows.append({"pair": pid, "table": "*", "status": "ERROR", "issues": f"source: {err}"})
    for pid, err in dest_db_errors.items():
        summary_rows.append({"pair": pid, "table": "*", "status": "ERROR", "issues": f"dest: {err}"})

    for key in sorted(set(src) | set(dest)):
        pid, table = key
        s, d = src.get(key), dest.get(key)
        if d is None and pid in dest_db_errors:
            continue  # already reported once for the whole database
        result, diffs = compare_table(s, d)
        notes = [f"source: {n}" for n in (s or {}).get("notes", [])]
        notes += [f"dest: {n}" for n in (d or {}).get("notes", [])]
        sp, dp = (s or {}).get("registered_partitions"), (d or {}).get("registered_partitions")
        summary_rows.append({
            "pair": pid,
            "table": (s or d)["table"],
            **result,
            "source_rows": (s or {}).get("row_count"),
            "dest_rows": (d or {}).get("row_count"),
            "source_partitions": None if sp is None else len(sp),
            "dest_partitions": None if dp is None else len(dp),
            "source_files": (s or {}).get("data_file_count"),
            "dest_files": (d or {}).get("data_file_count"),
            "source_bytes": (s or {}).get("data_bytes"),
            "dest_bytes": (d or {}).get("data_bytes"),
            "partition_issues": len(diffs),
            "notes": "; ".join(notes),
            "source_location": (s or {}).get("location"),
            "dest_location": (d or {}).get("location"),
        })
        diff_rows += [{"pair": pid, "table": (s or d)["table"], **row} for row in diffs]

    summary_cols = [
        "pair", "table", "status", "issues", "source_rows", "dest_rows", "source_partitions", "dest_partitions",
        "source_files", "dest_files", "source_bytes", "dest_bytes", "partition_issues", "schema_differences",
        "partition_schema_differences", "notes", "source_location", "dest_location",
    ]
    diff_cols = ["pair", "table", "partition", "issue", "source_rows", "dest_rows",
                 "source_files", "dest_files", "source_bytes", "dest_bytes"]
    summary = pd.DataFrame(summary_rows).reindex(columns=summary_cols)
    diffs = pd.DataFrame(diff_rows).reindex(columns=diff_cols)
    for df, cols in ((summary, summary_cols[4:13]), (diffs, diff_cols[4:])):
        for col in cols:
            df[col] = df[col].astype("Int64")
    severity = ["ERROR", "FAIL", "MISSING_ON_DEST", "WARN", "EXTRA_ON_DEST", "SKIPPED_VIEW", "PASS"]
    status_order = {s: i for i, s in enumerate(severity)}
    summary = summary.sort_values(
        ["status", "pair", "table"], key=lambda c: c.map(status_order) if c.name == "status" else c
    )
    return summary.reset_index(drop=True), diffs

In [ ]:
_texts = {side: read_text(snapshot_path(side)) for side in ("source", "dest")}
missing = [side for side, text in _texts.items() if text is None]
summary_df = partitions_df = None
if missing:
    print(f"Comparison skipped: no snapshot yet for {missing} under {RUN_DIR}.")
    print("Collect that side (on its platform) with the same OUTPUT_DIR and RUN_LABEL, then re-run this cell.")
else:
    src_snapshot, dest_snapshot = json.loads(_texts["source"]), json.loads(_texts["dest"])
    for snap in (src_snapshot, dest_snapshot):
        print(f"{snap['side']:>6}: {snap['label']} collected {snap['collected_at']} (Spark {snap['spark_version']})")
    summary_df, partitions_df = compare_snapshots(src_snapshot, dest_snapshot)
    print()
    print(summary_df["status"].value_counts().to_string())
    display(summary_df)

---
## Step 3: Write the report

Writes `summary.csv`, `partition_diffs.csv` and `report.html` to a new `report_<UTC timestamp>/` folder.

In [ ]:
STATUS_COLORS = {
    "PASS": "#d4edda", "WARN": "#fff3cd", "FAIL": "#f8d7da", "ERROR": "#f8d7da",
    "MISSING_ON_DEST": "#f8d7da", "EXTRA_ON_DEST": "#e2e3e5", "SKIPPED_VIEW": "#e2e3e5",
}


def _html_table(df):
    shown = df.copy()
    for col in [c for c in shown.columns if c.endswith("_bytes")]:
        shown[col] = shown[col].map(fmt_bytes)
    out = shown.to_html(index=False, na_rep="", escape=True, border=0)
    for status, color in STATUS_COLORS.items():
        out = out.replace(f"<td>{status}</td>", f'<td style="background:{color};font-weight:bold">{status}</td>')
    return out


def render_report(summary, diffs, src_snap, dest_snap):
    counts = " &nbsp; ".join(f"<b>{html.escape(k)}</b>: {v}" for k, v in summary["status"].value_counts().items())
    shown_note = ""
    if len(diffs) > REPORT_MAX_PARTITION_ROWS:
        shown_note = f", first {REPORT_MAX_PARTITION_ROWS} shown, see partition_diffs.csv"
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Migration validation: {html.escape(RUN_LABEL)}</title>
<style>
 body {{ font-family: sans-serif; font-size: 13px; margin: 24px; }}
 table {{ border-collapse: collapse; margin-bottom: 24px; }}
 th, td {{ border: 1px solid #ccc; padding: 4px 8px; vertical-align: top; }}
 th {{ background: #f2f2f2; }}
</style></head><body>
<h1>Migration validation: {html.escape(src_snap['label'])} &rarr; {html.escape(dest_snap['label'])}</h1>
<p>Run label <b>{html.escape(RUN_LABEL)}</b>.
 Source collected {html.escape(src_snap['collected_at'])}, dest collected {html.escape(dest_snap['collected_at'])}.</p>
<p>{counts}</p>
<h2>Tables</h2>
{_html_table(summary)}
<h2>Partition differences ({len(diffs):,}{shown_note})</h2>
{_html_table(diffs.head(REPORT_MAX_PARTITION_ROWS)) if len(diffs) else "<p>None.</p>"}
</body></html>"""


if summary_df is None:
    print("No comparison to report; see Step 2.")
else:
    report_dir = f"{RUN_DIR}/report_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    write_text(f"{report_dir}/summary.csv", summary_df.to_csv(index=False))
    write_text(f"{report_dir}/partition_diffs.csv", partitions_df.to_csv(index=False))
    write_text(f"{report_dir}/report.html", render_report(summary_df, partitions_df, src_snapshot, dest_snapshot))
    print(f"Report written to {report_dir}/")

---
## Drill-down: one table

Lists every partition-level difference for one table: rows per partition, file count and bytes
per partition directory, and partition directories that are not registered.

In [ ]:
TABLE_TO_INSPECT = ""  # table name, e.g. "orders"

if TABLE_TO_INSPECT and summary_df is not None:
    _mask = summary_df["table"].str.lower() == TABLE_TO_INSPECT.lower()
    display(summary_df[_mask].T)
    display(partitions_df[partitions_df["table"].str.lower() == TABLE_TO_INSPECT.lower()])